In [1]:
import json
import random
from datetime import datetime, timedelta
from ucimlrepo import fetch_ucirepo, list_available_datasets
import numpy as np
import pandas as pd
from etl import UserGenerator
from feature_engineer import FeatureEngineer
from sklearn.linear_model import LogisticRegression
from train_mlflow import TrainMlflow
from train_mlflow_advance import TrainOptuna

c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# ETL

In [2]:
user_generator = UserGenerator(n_samples=25000)


In [3]:
ds = user_generator.create_dataset()
print(type(ds), isinstance(ds, tuple))

<class 'pandas.core.frame.DataFrame'> False


In [4]:
ds

,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...
541904,PACK OF 20 SPACEBOY NAPKINS,12,12/9/2011 12:50,0.85,12680.0,France
541905,CHILDREN'S APRON DOLLY GIRL,6,12/9/2011 12:50,2.10,12680.0,France
541906,CHILDRENS CUTLERY DOLLY GIRL,4,12/9/2011 12:50,4.15,12680.0,France
541907,CHILDRENS CUTLERY CIRCUS PARADE,4,12/9/2011 12:50,4.15,12680.0,France


In [5]:
ds = user_generator.run_etl()

In [6]:
ds.info

<bound method DataFrame.info of                                 Description  Quantity         InvoiceDate  \
0        WHITE HANGING HEART T-LIGHT HOLDER         6 2010-01-12 08:26:00   
1                       WHITE METAL LANTERN         6 2010-01-12 08:26:00   
2            CREAM CUPID HEARTS COAT HANGER         8 2010-01-12 08:26:00   
3       KNITTED UNION FLAG HOT WATER BOTTLE         6 2010-01-12 08:26:00   
4            RED WOOLLY HOTTIE WHITE HEART.         6 2010-01-12 08:26:00   
...                                     ...       ...                 ...   
541904          PACK OF 20 SPACEBOY NAPKINS        12 2011-09-12 12:50:00   
541905          CHILDREN'S APRON DOLLY GIRL         6 2011-09-12 12:50:00   
541906         CHILDRENS CUTLERY DOLLY GIRL         4 2011-09-12 12:50:00   
541907      CHILDRENS CUTLERY CIRCUS PARADE         4 2011-09-12 12:50:00   
541908         BAKING SET 9 PIECE RETROSPOT         3 2011-09-12 12:50:00   

        UnitPrice  CustomerID         Count

In [7]:
ds.describe().T

,count,mean,min,25%,50%,75%,max,std
Quantity,168631.0,13.079546,1.0,2.0,6.0,12.0,80995.0,201.856371
InvoiceDate,168631,2011-05-16 06:45:10.094585088,2010-01-12 08:26:00,2011-03-05 12:10:00,2011-06-09 09:51:00,2011-09-06 13:08:00,2011-12-10 17:19:00,NaN
UnitPrice,168631.0,3.147421,0.04,1.25,1.95,3.75,8142.75,26.125811
CustomerID,168631.0,15295.174292,12347.0,13881.0,15192.0,16873.0,18287.0,1730.186972


In [8]:
ds.columns

Index(['Description', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID',
       'Country'],
      dtype='object')

In [9]:
# Información base
print("Fecha mínima:", ds["InvoiceDate"].min())
print("Fecha máxima:", ds["InvoiceDate"].max())
print("Clientes únicos:", ds["CustomerID"].nunique())
print("Productos únicos:", ds["Description"].nunique())
print("Países:", ds["Country"].nunique())
ds["InvoiceDate"] = pd.to_datetime(ds["InvoiceDate"], errors="coerce")
print("Productos únicos:", ds["Description"].nunique())
print("Países:", ds["Country"].nunique())
print(f"Rango de fechas: {ds['InvoiceDate'].min().date()} → {ds['InvoiceDate'].max().date()}")


Fecha mínima: 2010-01-12 08:26:00
Fecha máxima: 2011-12-10 17:19:00
Clientes únicos: 2997
Productos únicos: 3671
Países: 33
Productos únicos: 3671
Países: 33
Rango de fechas: 2010-01-12 → 2011-12-10


# Feature Engineering

In [10]:
feature_engineer = FeatureEngineer(ds)

In [11]:
df_engineered = feature_engineer.run()


In [12]:
df_engineered

,InvoiceDate,Quantity,Revenue,UnitPrice,Country,CustomerID,n_past_invoices,prev_date,recency_days,spend_prior,qty_prior,avg_ticket_prior,avg_qty_per_invoice_prior,next_date,days_to_next,y_repurchase_30d
0,2010-07-12 14:57:00,319.0,711.79,2.890000,Iceland,12347,0.0,NaT,9999.0,0.00,0.0,0.000000,0.000000,2011-02-08 08:48:00,210.0,0
1,2011-02-08 08:48:00,277.0,584.91,3.101818,Iceland,12347,1.0,2010-07-12 14:57:00,210.0,711.79,319.0,711.790000,319.000000,2011-07-04 10:43:00,146.0,0
2,2011-07-04 10:43:00,483.0,636.25,2.595417,Iceland,12347,2.0,2011-02-08 08:48:00,146.0,1296.70,596.0,648.350000,298.000000,2011-07-12 15:52:00,8.0,1
3,2011-07-12 15:52:00,192.0,224.82,1.230909,Iceland,12347,3.0,2011-07-04 10:43:00,8.0,1932.95,1079.0,644.316667,359.666667,2011-09-06 13:01:00,55.0,0
4,2011-09-06 13:01:00,196.0,382.52,2.978889,Iceland,12347,4.0,2011-07-12 15:52:00,55.0,2157.77,1271.0,539.442500,317.750000,NaT,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7353,2011-05-09 12:35:00,95.0,134.90,1.401915,United Kingdom,18283,0.0,NaT,9999.0,0.00,0.0,0.000000,0.000000,2011-06-01 14:14:00,23.0,1
7354,2011-06-01 14:14:00,61.0,108.45,1.771053,United Kingdom,18283,1.0,2011-05-09 12:35:00,23.0,134.90,95.0,134.900000,95.000000,2011-06-12 12:02:00,10.0,1
7355,2011-06-12 12:02:00,142.0,208.00,1.307600,United Kingdom,18283,2.0,2011-06-01 14:14:00,10.0,243.35,156.0,121.675000,78.000000,2011-10-11 14:59:00,121.0,0
7356,2011-10-11 14:59:00,64.0,112.35,1.757627,United Kingdom,18283,3.0,2011-06-12 12:02:00,121.0,451.35,298.0,150.450000,99.333333,2011-10-11 15:07:00,0.0,1


# Modelando con MLFlow

In [13]:
import mlflow


experiment_name = "recompra-LogReg"
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment(experiment_name)

# Enable autologging for sklearn models
mlflow.sklearn.autolog(
    log_input_examples=True,
    log_model_signatures=True,
    log_models=True,
    disable=False,
    exclusive=False,
    disable_for_unsupported_versions=False,
    silent=False,
    max_tuning_runs=5
)

In [14]:
num_feats = [
    'recency_days','n_past_invoices','spend_prior','qty_prior',
    'avg_ticket_prior','avg_qty_per_invoice_prior','UnitPrice','Quantity','Revenue'
]
cat_feats = ['Country']

In [15]:


model = LogisticRegression(max_iter=500)

# 3) Instancia y entrena
trainer = TrainMlflow(
    df=df_engineered,
    numeric_features=num_feats,
    categorical_features=cat_feats,
    target_column='y_repurchase_30d',
    model=model,
    mlflow_setup={"tracking_uri": "file:./mlruns", "experiment_name": "OnlineRetail"}
)

pipeline, run_id = trainer.train()
trainer.pipeline = pipeline                  # <- necesario para save_model()
trainer.save_model("models/model.pkl")       # ✅ Modelo guardado en models/model.pkl


Rango total: 2010-01-12 08:26:00 → 2011-11-10 16:50:00 | cutoff: 2011-10-11 16:50:00
train_end: 2011-09-01 00:00:00
train: 3217 | test: 820
pos_rate train=0.417 | test=0.359


MLflow Run ID: ca17ab948fce47279933fa168baeb641
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.7097
Test Accuracy: 0.6976
🏃 View run classy-kit-921 at: http://127.0.0.1:5000/#/experiments/691171428289487265/runs/ca17ab948fce47279933fa168baeb641
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/691171428289487265
✅ Modelo guardado en models/model.pkl


'models/model.pkl'

In [16]:
mlflow.set_experiment("recompra-optuna")

params = {
            'solver': ('categorical', ['lbfgs', 'liblinear', 'saga']),
            'C':      ('float', 1e-3, 1e2, True),
            'max_iter': ('int', 300, 1500),
            'class_weight': ('categorical', [None, 'balanced']),
            # solver-specific penalties are tricky to encode generically—start simple with l2
            'penalty': ('categorical', ['l2']),
}

trainer = TrainOptuna(
    df=df_engineered,
    numeric_features=num_feats,
    categorical_features=cat_feats,
    target_column='y_repurchase_30d',
    model_class=LogisticRegression,
    model_params={},                 
    n_trials=30,                     
    optimization_metric='roc_auc',   
    param_distributions=params,
)

best_pipeline, best_run_id, study = trainer.train()   # runs Optuna + logs to MLflow
trainer.save_model("models/modeloptuna.pkl")


[I 2025-10-05 11:36:29,440] A new study created in memory with name: optuna_LogisticRegression


Rango total: 2010-01-12 08:26:00 → 2011-11-10 16:50:00 | cutoff: 2011-10-11 16:50:00
train_end: 2011-09-01 00:00:00
train: 3217 | test: 820
pos_rate train=0.417 | test=0.359
Starting Optuna optimization with 30 trials...
Optimizing for: roc_auc
Model type: LogisticRegression


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\src\app\train\train_mlflow_advance.py:248: ExperimentalWarning: MLflowCallback is experimental (supported from v1.4.0). The interface can change in the future.
  mlflow_callback = MLflowCallback(
2025/10/05 11:36:29 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'ba6a3eae056748ee9e956503d71b7bc5', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow
[I 2025-10-05 11:36:35,475] Trial 0 finished with value: 0.7283244096117535 and parameters: {'solver': 'lbfgs', 'C': 0.1237759717923414, 'max_iter': 412, 'class_weight': None, 'penalty': 'l2'}. Best is trial 0 with value: 0.7283244096117535.


🏃 View run resilient-rat-486 at: http://127.0.0.1:5000/#/experiments/279235497303288747/runs/ba6a3eae056748ee9e956503d71b7bc5
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/279235497303288747


2025/10/05 11:36:35 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID '9f12693963fc411ba0380ff8b117ac50', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current sklearn workflow


...

[I 2025-10-05 11:42:43,815] Trial 29 finished with value: 0.7288481932697034 and parameters: {'solver': 'liblinear', 'C': 0.18669169512168524, 'max_iter': 489, 'class_weight': None, 'penalty': 'l2'}. Best is trial 15 with value: 0.7310338584102842.


🏃 View run likeable-lamb-254 at: http://127.0.0.1:5000/#/experiments/272927137381038857/runs/a28b557b23f44b119d319f9864c6910d
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/272927137381038857
🏃 View run 29 at: http://127.0.0.1:5000/#/experiments/272927137381038857/runs/d00ee2f2c35b4bc4b5f70e836442df2e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/272927137381038857

Optimization complete!
Best roc_auc: 0.7310
Best parameters: {'solver': 'saga', 'C': 18.943341777540624, 'max_iter': 346, 'class_weight': 'balanced', 'penalty': 'l2'}


c:\Users\gabri\OneDrive\ProyectoFinalMLOps\.venv\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
2025/10/05 11:42:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Best Model MLflow Run ID: 180b8816dccd4730832d2f29afad128e
Tracking URI: http://127.0.0.1:5000
Train Accuracy: 0.7118
Test Accuracy: 0.6841
🏃 View run best_model_LogisticRegression at: http://127.0.0.1:5000/#/experiments/272927137381038857/runs/180b8816dccd4730832d2f29afad128e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/272927137381038857
✅ Modelo guardado en models/modeloptuna.pkl


'models/modeloptuna.pkl'